# 📡 Servicios

**Python, Datos e Ingeniería de IA Aplicada**  
UTN Rosario — FRRO, Área de Extensión Universitaria — 2026

**Encuentro 2 · Bloque 3 — 30 minutos**

---

## Objetivo

Conectar el programa con el mundo exterior: aprender a usar módulos de la biblioteca estándar de Python y a consumir APIs públicas para incorporar datos reales a cualquier script.

## Al terminar este bloque van a poder:

1. Importar y usar módulos de la biblioteca estándar de Python
2. Hacer peticiones HTTP a una API pública con la librería `requests`
3. Navegar y procesar la respuesta JSON de una API para extraer información útil

---

## ◈ Microglosario

| Término | Qué es en lenguaje llano |
|---|---|
| **Módulo** | Un archivo de Python que contiene funciones y herramientas ya escritas. Lo importamos para no reinventar la rueda. |
| **Biblioteca estándar** | El conjunto de módulos que viene incluido con Python sin instalar nada extra. |
| **API** | Una interfaz que permite que dos programas se comuniquen. Pedimos datos, el servidor los devuelve. |
| **Endpoint** | La URL específica dentro de una API que responde a un tipo de pedido. |
| **HTTP GET** | El tipo de petición más común: pedimos datos sin modificar nada en el servidor. |
| **Status code** | El número que devuelve el servidor para indicar si la petición funcionó (`200` = ok, `404` = no encontrado, `500` = error del servidor). |

---

## ⚙️ Módulos

### Analogía

Los módulos son cajas de herramientas especializadas. No traés toda la ferretería a la obra: traés la caja de electricidad cuando trabajás con cables, y la caja de plomería cuando trabajás con caños. `import` es la instrucción que le dice a Python qué caja traer.

### Dónde vive esto en el mundo real

Python tiene más de 300 módulos en su biblioteca estándar — y miles más instalables. Todo proyecto profesional usa decenas. En este curso ya usamos `json` (Bloque 2). En los próximos encuentros vamos a sumar `pathlib` para archivos, `fastapi` para el backend, `pandas` para datos, y las librerías de IA. Saber importar y encontrar módulos es una habilidad permanente: la mayor parte del tiempo el código ya existe, hay que saber dónde buscarlo.

In [ ]:
# Importamos módulos de la biblioteca estándar — no necesitan instalación
import os        # interacción con el sistema de archivos y el sistema operativo
import datetime  # fechas, horas y diferencias de tiempo
import json      # ya lo conocemos del bloque anterior

print("✓ Módulos de la biblioteca estándar cargados.")

In [ ]:
# os nos da acceso al sistema de archivos
directorio_actual = os.getcwd()          # directorio de trabajo actual
archivos_en_carpeta = os.listdir(".")    # archivos y carpetas en el directorio actual

print(f"Directorio actual: {directorio_actual}")
print()
print("Archivos en esta carpeta:")
for nombre_archivo in archivos_en_carpeta:
    print(f"  · {nombre_archivo}")

In [ ]:
# datetime nos permite trabajar con fechas y horas
hoy = datetime.date.today()
ahora = datetime.datetime.now()

print(f"Fecha de hoy   : {hoy}")
print(f"Fecha y hora   : {ahora.strftime('%d/%m/%Y %H:%M')}")

# Calculamos cuántos días faltan para el Demo Day (Encuentro 11)
# Ajustá la fecha según el calendario real de tu cursada
demo_day = datetime.date(2026, 8, 26)
dias_restantes = (demo_day - hoy).days

print(f"Días para el Demo Day: {dias_restantes}")

### ✎ Para pensar

- ¿Qué diferencia hay entre `import datetime` y `from datetime import date`?
- ¿Qué otros módulos de la biblioteca estándar conocen o usaron antes?
  Podés explorar la lista en [docs.python.org/3/library](https://docs.python.org/3/library/).

---

## 🔗 La librería `requests`

### Analogía

Una API es como un mozo en un restaurante: vos pedís el plato (el dato que necesitás), el mozo va a la cocina (el servidor), y vuelve con el resultado. No necesitás entrar a la cocina, ni saber cómo se preparó, ni conocer la receta. Solo necesitás saber cómo pedirlo correctamente.

`requests` es la librería que hace ese pedido por nosotros.

### Dónde vive esto en el mundo real

Las APIs son la infraestructura de la economía digital. La temperatura que ves en tu celular viene de una API. El precio del dólar en una app de finanzas viene de una API. Los datos de lluvia que usa un sistema de riego automatizado en el agro vienen de una API. Cuando en el Encuentro 4 construyan su propio backend con FastAPI, van a estar del otro lado: van a ser el servidor que responde los pedidos.

In [ ]:
# requests no viene en la biblioteca estándar: hay que instalarlo
# En la terminal: pip install requests
# En este notebook podemos instalarlo directamente con:
!uv pip install requests --quiet

import requests

print("✓ Librería requests disponible.")

In [ ]:
# Hacemos una petición simple para verificar que la conexión funciona
# httpbin.org es un servicio de prueba que devuelve información sobre nuestra petición
url_prueba = "https://httpbin.org/get"

respuesta = requests.get(url_prueba)

# El status code indica si la petición fue exitosa
print(f"Status code : {respuesta.status_code}")
print(f"200 = OK    : {respuesta.status_code == 200}")

Los códigos de estado HTTP más importantes:

| Código | Significado |
|---|---|
| `200` | OK — la petición funcionó |
| `400` | Bad Request — el pedido tiene errores |
| `401` | Unauthorized — falta autenticación |
| `404` | Not Found — la URL no existe |
| `429` | Too Many Requests — excedimos el límite de peticiones |
| `500` | Internal Server Error — el servidor falló |

---

## 🌾 Consultando el clima para el campo

Vamos a usar [Open-Meteo](https://open-meteo.com/): una API meteorológica de código abierto, **gratuita y sin API key**. Devuelve pronósticos de hasta 16 días con variables relevantes para la agricultura: temperatura, precipitación, viento y evapotranspiración.

El caso: una cooperativa agrícola de la región de Rosario quiere automatizar la consulta del pronóstico semanal para planificar las tareas de campo.

In [ ]:
# Coordenadas de Rosario, Argentina
latitud = -32.9468
longitud = -60.6393

# Variables climáticas relevantes para la planificación agrícola
variables_diarias = "temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max"

# Construimos la URL de la petición paso a paso
url_base = "https://api.open-meteo.com/v1/forecast"
parametros = (
    f"latitude={latitud}"
    f"&longitude={longitud}"
    f"&daily={variables_diarias}"
    f"&timezone=America%2FArgentina%2FBuenos_Aires"
    f"&forecast_days=7"
)
url_completa = f"{url_base}?{parametros}"

print("✓ URL construida.")
print(f"   {url_completa[:80]}...")

In [ ]:
# Realizamos la petición con manejo de errores
try:
    respuesta = requests.get(url_completa, timeout=10)    # timeout evita esperar indefinidamente
    respuesta.raise_for_status()                           # lanza excepción si el status no es 200

    datos_clima = respuesta.json()                         # convertimos la respuesta a diccionario
    print(f"✓ Datos recibidos. Status: {respuesta.status_code}")

except requests.exceptions.ConnectionError:
    print("✗ Sin conexión a Internet.")

except requests.exceptions.Timeout:
    print("✗ La petición tardó demasiado. Intentá de nuevo.")

except requests.exceptions.HTTPError as error:
    print(f"✗ Error del servidor: {error}")

In [ ]:
# Exploramos la estructura de la respuesta para entender qué recibimos
claves_principales = list(datos_clima.keys())
print("Claves en la respuesta:")
for clave in claves_principales:
    print(f"  · {clave}")

In [ ]:
# Los datos diarios están en la clave 'daily'
datos_diarios = datos_clima["daily"]

# Extraemos cada variable en su propia lista
fechas           = datos_diarios["time"]
temp_maxima      = datos_diarios["temperature_2m_max"]
temp_minima      = datos_diarios["temperature_2m_min"]
precipitacion    = datos_diarios["precipitation_sum"]
viento_maximo    = datos_diarios["wind_speed_10m_max"]

print(f"✓ Pronóstico para {len(fechas)} días extraído.")

In [ ]:
# Mostramos el pronóstico en formato tabular
print(f"{'Fecha':<12} {'Máx (°C)':>8} {'Mín (°C)':>8} {'Lluvia (mm)':>11} {'Viento (km/h)':>13}")
print("─" * 57)

for indice in range(len(fechas)):
    fecha_actual  = fechas[indice]
    max_actual    = temp_maxima[indice]
    min_actual    = temp_minima[indice]
    lluvia_actual = precipitacion[indice]
    viento_actual = viento_maximo[indice]

    print(f"{fecha_actual:<12} {max_actual:>8.1f} {min_actual:>8.1f} {lluvia_actual:>11.1f} {viento_actual:>13.1f}")

In [ ]:
# Identificamos días con condiciones problemáticas para el campo
# Criterios simples de alerta agroclimática
umbral_lluvia  = 10.0    # mm — por encima puede dificultar las tareas
umbral_viento  = 40.0    # km/h — por encima afecta la fumigación
umbral_calor   = 35.0    # °C — estrés térmico en cultivos

print("── Alertas agroclimáticas de la semana ───────────────")
hay_alertas = False

for indice in range(len(fechas)):
    alertas_del_dia = []    # acumulamos las alertas de cada día

    if precipitacion[indice] > umbral_lluvia:
        alertas_del_dia.append(f"lluvia {precipitacion[indice]:.0f} mm")

    if viento_maximo[indice] > umbral_viento:
        alertas_del_dia.append(f"viento {viento_maximo[indice]:.0f} km/h")

    if temp_maxima[indice] > umbral_calor:
        alertas_del_dia.append(f"calor {temp_maxima[indice]:.0f}°C")

    # Solo mostramos los días que tienen al menos una alerta
    if len(alertas_del_dia) > 0:
        hay_alertas = True
        alertas_unidas = " · ".join(alertas_del_dia)
        print(f"  ▲ {fechas[indice]}: {alertas_unidas}")

if not hay_alertas:
    print("  ✓ Sin alertas esta semana. Condiciones favorables.")

In [ ]:
# Guardamos el pronóstico en un archivo JSON para usarlo en el integrador
pronostico_para_guardar = []

for indice in range(len(fechas)):
    registro_diario = {
        "fecha": fechas[indice],
        "temp_max": temp_maxima[indice],
        "temp_min": temp_minima[indice],
        "lluvia_mm": precipitacion[indice],
        "viento_kmh": viento_maximo[indice]
    }
    pronostico_para_guardar.append(registro_diario)

with open("pronostico_rosario.json", "w", encoding="utf-8") as archivo:
    json.dump(pronostico_para_guardar, archivo, ensure_ascii=False, indent=2)

print(f"✓ Pronóstico guardado en pronostico_rosario.json ({len(pronostico_para_guardar)} días).")

### ✎ Para pensar

- ¿Qué ventaja tiene guardar el pronóstico en un archivo JSON en vez de consultarlo cada vez que lo necesitamos?
- ¿Cómo cambiarías el código para consultar el clima de otra ciudad o región? ¿Cuántas líneas hay que modificar?
- ¿Qué otras variables climáticas de Open-Meteo podrían ser útiles para la planificación agrícola?

In [ ]:
# ─── Espacio de práctica ──────────────────────────────────────────────────────
#
# 1. Modificá el código para consultar el clima de otra ciudad argentina
#    que les resulte relevante (Córdoba, Mendoza, Buenos Aires, etc.)
#    Podés encontrar coordenadas en Google Maps.
#
# 2. Escribí una función llamada mejor_dia_para_fumigar(pronostico) que:
#    - Reciba la lista de registros diarios
#    - Descarte los días con lluvia > 5 mm o viento > 30 km/h
#    - Devuelva la fecha del mejor día disponible
#    - Si no hay días favorables, devuelva None
#
# Bonus: armá un informe de texto que incluya ciudad, fecha de consulta
#         y el resultado de mejor_dia_para_fumigar().
#
# ─────────────────────────────────────────────────────────────────────────────

---

## 🌾 Cierre del bloque

El programa ya no está solo. Aprendimos a:

- **Importar módulos** → usar herramientas que ya existen en lugar de escribirlas desde cero
- **Consumir una API** → pedir datos a un servidor externo con `requests.get()`
- **Procesar la respuesta** → navegar JSON anidado y extraer la información que necesitamos
- **Combinar bloques** → tipos + estructuras de control + funciones + archivos + APIs, todo junto

En el siguiente bloque vamos a ensamblar todo esto en un script integrador que se commitea al repositorio de la cursada.

---

*Próximo bloque → `004_integrador` — Ensamblado y commit al repo*